In [ ]:
import pandas as pd
from pathlib import Path

In [12]:


df_original = pd.read_csv("DenunciasPoliciales_1000000.csv")
print("Dimensiones del dataset:", df_original.shape)


Dimensiones del dataset: (1000000, 8)


In [6]:
df_original.head()

,ANIO,MES,DPTO_HECHO_NEW,PROV_HECHO,DIST_HECHO,UBIGEO_HECHO,P_MODALIDADES,cantidad
0,2018,1,AMAZONAS,BAGUA,ARAMANGO,10202,Otros,4
1,2018,1,AMAZONAS,BAGUA,ARAMANGO,10202,Violencia contra la mujer e integrantes,3
2,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Estafa,2
3,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Hurto,25
4,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Otros,52


In [13]:
#Creando copia del dataset

df = df_original.copy()

In [14]:
display(df.dtypes.to_frame("tipo"))
display(df.isna().sum().to_frame("nulos"))
print("Filas duplicadas exactas:", df.duplicated().sum())


,tipo
ANIO,int64
MES,int64
DPTO_HECHO_NEW,object
PROV_HECHO,object
DIST_HECHO,object
UBIGEO_HECHO,int64
P_MODALIDADES,object
cantidad,int64


,nulos
ANIO,0
MES,0
DPTO_HECHO_NEW,0
PROV_HECHO,0
DIST_HECHO,0
UBIGEO_HECHO,0
P_MODALIDADES,0
cantidad,0


Filas duplicadas exactas: 0


In [15]:
columnas_ubicacion = ["DPTO_HECHO_NEW", "PROV_HECHO", "DIST_HECHO"]
columnas_texto = columnas_ubicacion + ["P_MODALIDADES"]
for col in columnas_texto:
    df[col] = (df[col].astype("string").str.strip()
               .str.replace(r"\s+", " ", regex=True)
               .replace("", pd.NA))
for col in columnas_ubicacion:
    df[col] = df[col].str.upper()

display(df[columnas_texto].head())

,DPTO_HECHO_NEW,PROV_HECHO,DIST_HECHO,P_MODALIDADES
0,AMAZONAS,BAGUA,ARAMANGO,Otros
1,AMAZONAS,BAGUA,ARAMANGO,Violencia contra la mujer e integrantes
2,AMAZONAS,BAGUA,BAGUA,Estafa
3,AMAZONAS,BAGUA,BAGUA,Hurto
4,AMAZONAS,BAGUA,BAGUA,Otros


In [16]:
#Confirmando si los valores tienen coherencia

for col in ["ANIO", "MES", "cantidad"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["UBIGEO_HECHO"] = (df["UBIGEO_HECHO"].astype("string")
    .str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(6))

validos = (
    df["ANIO"].between(2015, 2026) & (df["ANIO"] % 1 == 0) &
    df["MES"].between(1, 12) & (df["MES"] % 1 == 0) &
    df["cantidad"].ge(0) & (df["cantidad"] % 1 == 0) &
    df["UBIGEO_HECHO"].str.fullmatch(r"\d{6}").fillna(False) &
    df[columnas_texto].notna().all(axis=1)
).fillna(False)

print("Registros con valores incompletos o inválidos:", (~validos).sum())
df_invalidos = df.loc[~validos].copy()
df = df.loc[validos].copy()
for col in ["ANIO", "MES", "cantidad"]:
    df[col] = df[col].astype("int64")

Registros con valores incompletos o inválidos: 0


In [17]:
duplicados = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicados exactos eliminados:", duplicados)

Duplicados exactos eliminados: 0


In [18]:
archivo_nuevo = "DenunciasPoliciales_limpio.csv"
df.to_csv(archivo_nuevo, index=False, encoding="utf-8-sig")
if not df_invalidos.empty:
    df_invalidos.to_csv("DenunciasPoliciales_registros_invalidos.csv", index=False, encoding="utf-8-sig")
print("Archivo generado:", archivo_nuevo)

Archivo generado: DenunciasPoliciales_limpio.csv
